# BÁO CÁO GIỮA KỲ: Đề xuất và cài đặt phiên bản tuần tự

**Khoa:** Công nghệ thông tin

**Trường:** Đại học Khoa học tự nhiên, ĐHQG-HCM

**GVHD:** thầy Phạm Trọng Nghĩa

**Nhóm:** 07

**Năm học:** 2026 – 2027

---

## 1. Thông tin nhóm và kế hoạch công việc

### 1.1 Danh sách thành viên

| STT | MSSV      | Họ và tên             | Vai trò |
|:---:|-----------|-----------------------|---------|
| 1   | 23120315  | Nguyễn Đăng Pha       |         |
| 2   | 23120356  | Trần Nguyễn Trí Thanh |         |
| 3   | 23120360  | Đặng Lê Đức Thịnh     |         |
| 4   | 23120366  | Trương Thanh Thuận    |         |

### 1.2 Kế hoạch phân công

| Giai đoạn - Công việc                                       | Thành viên phụ trách   | Phiên bản  |
|-------------------------------------------------------------|------------------------|------------|
| Giai đoạn 1 - Tiền xử lý (Preprocessing)                    |                        | V0         |
| Giai đoạn 2 - Phát hiện điểm đặc trưng (Keypoint Detection) |                        | V0         |
| Giai đoạn 3 - Mô tả điểm đặc trưng (Description)            |                        | V0         |
| Giai đoạn 4 - Ghép cặp điểm (Matching)                      |                        | V0         |
| Giai đoạn 5 - Ước lượng Homography                          |                        | V0         |
| Giai đoạn 6 - Biến dạng và hòa trộn (Warping & Blending)    |                        | V0         |
| Đánh giá hiệu năng và báo cáo                               |                        | Tất cả     |

---

## 2. Hiểu vấn đề

### 2.1 Bài toán ghép ảnh toàn cảnh (Panorama Stitching)

Ghép ảnh toàn cảnh là quá trình kết hợp hai (hoặc nhiều) ảnh chụp có vùng chồng lấp của cùng một cảnh vật thành một ảnh góc rộng duy nhất...

**Đầu vào:** Hai ảnh RGB có vùng chồng lấp (ảnh A và ảnh B).  
**Đầu ra:** Một ảnh toàn cảnh duy nhất (ảnh B được biến dạng và hòa vào khung của ảnh A).

### 2.2 Pipeline 6 giai đoạn

| #   | Giai đoạn                    | Mô tả                                                                         |
|:---:|------------------------------|-------------------------------------------------------------------------------|
| 1   | **Tiền xử lý**               | Chuyển ảnh sang ảnh xám (grayscale) và làm mịn để giảm nhiễu                  |
| 2   | **Phát hiện điểm đặc trưng** | Tìm các điểm nổi bật (góc cạnh) trong từng ảnh                                |
| 3   | **Mô tả điểm đặc trưng**     | Tính vector mô tả nhỏ gọn cho vùng lân cận mỗi điểm                           |
| 4   | **Ghép cặp điểm**            | Với mỗi điểm ở ảnh A, tìm điểm tương đồng nhất ở ảnh B; loại bỏ các cặp mơ hồ |
| 5   | **Ước lượng Homography**     | Tính ma trận biến đổi xạ ảnh 3×3 để ánh xạ ảnh B sang ảnh A, xử lý nhiễu      |
| 6   | **Biến dạng và hòa trộn**    | Biến dạng ảnh B vào hệ tọa độ của ảnh A và hòa trộn vùng chồng lấp            |

---

## 3. Phương pháp được chọn

Nhóm chọn phương pháp cho từng giai đoạn dựa trên hai tiêu chí: **chất lượng đầu ra** và **khả năng tăng tốc trên GPU**.

<table>
  <thead>
    <tr>
      <th>Giai đoạn</th>
      <th>Phương pháp lựa chọn</th>
      <th>Cơ chế hoạt động</th>
      <th>Tham số chính</th>
      <th>Ưu thế chất lượng</th>
      <th>Tính phù hợp với GPU</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>1. Tiền xử lý</td>
      <td>Grayscale cố định + Separable Gaussian Blur (1D Horiz → 1D Vert)</td>
      <td>Chuyển RGB→Gray bằng công thức BT.601; nhân chập 1D theo hàng rồi theo cột thay vì kernel 2D đầy đủ</td>
      <td>σ (độ lệch chuẩn), k = 2⌈3σ⌉+1</td>
      <td>Loại bỏ nhiễu tần cao hiệu quả; O(2kN) thay vì O(k²N) mỗi pixel</td>
      <td>Mỗi pixel độc lập, hai pass 1D dễ map 1 thread/pixel; cache-friendly hơn kernel 2D</td>
    </tr>
    <tr>
      <td>2. Phát hiện điểm đặc trưng</td>
      <td>Harris Corner + NMS + Grid-based Top-K</td>
      <td>Tính gradient Ix/Iy bằng Sobel 3×3; làm mượt structure tensor; tính R = det(M) − k·trace(M)²; threshold, NMS rồi giữ Top-K theo từng ô lưới</td>
      <td>k = 0.04, σ_w = 1.0–1.5, R_thresh = 0.01·max(R), NMS 3×3, grid 32×32</td>
      <td>Response liên tục để xếp hạng; NMS loại cụm điểm; Grid Top-K giúp keypoint phân bố đều cho matching và homography</td>
      <td>Các bước là stencil/pixel-wise phù hợp 1 thread/pixel; có thể tối ưu bằng shared-memory tiling, reduction và parallel compaction</td>
    </tr>
    <tr>
      <td>3. Mô tả điểm đặc trưng</td>
      <td>ORB Descriptor (BRIEF có xoay, 256 bit)</td>
      <td>Lấy patch quanh keypoint trên ảnh đã làm mịn; tính hướng θ bằng Intensity Centroid; xoay tập 256 cặp điểm cố định theo θ; mỗi cặp so sánh cường độ I(x) &lt; I(y) cho ra 1 bit</td>
      <td>Patch S = 31×31, bán kính centroid r = 15, n = 256 bit, bước góc 12°</td>
      <td>Bất biến với phép xoay trong mặt phẳng (máy ảnh nghiêng khi chụp); làm mịn trước giúp các bit ít nhạy nhiễu; đủ dùng vì ảnh chụp xoay tại chỗ nên gần như không đổi tỉ lệ</td>
      <td>Mỗi keypoint độc lập (mỗi thread xử lý một keypoint), khối lượng việc đồng đều nên ít phân kỳ warp; mẫu cặp điểm cố định đặt trong Constant Memory (broadcast); descriptor 32 byte gọn, so khớp được bằng phép đếm bit</td>
    </tr>
    <tr>
      <td>4. Ghép cặp điểm</td>
      <td>Brute-force Hamming + Lowe Ratio Test + Cross-check</td>
      <td>Tính khoảng cách Hamming (XOR + đếm bit) giữa mọi cặp descriptor A–B; với mỗi điểm A giữ láng giềng gần nhất d₁ và gần nhì d₂; chấp nhận cặp nếu d₁/d₂ &lt; ρ và hai điểm là láng giềng gần nhất của nhau theo cả hai chiều</td>
      <td>ρ = 0.7 – 0.8, cross-check bật</td>
      <td>So khớp chính xác (không xấp xỉ như KD-tree/LSH); ratio test loại cặp mơ hồ ở vùng hoa văn lặp, cross-check loại cặp nhiều-một, giảm outlier đưa vào RANSAC</td>
      <td>Bài toán all-pairs tương tự nhân ma trận; mỗi khoảng cách chỉ cần XOR và đếm bit; dữ liệu được tái sử dụng nhiều nên hợp với Shared-memory Tiling; tìm d₁, d₂ là bài toán Reduction (warp shuffle)</td>
    </tr>
    <tr>
      <td>5. Ước lượng Homography</td>
      <td>RANSAC + 4-point DLT + Inlier Refinement</td>
      <td>Lấy mẫu ngẫu nhiên 4 cặp điểm không đồng tuyến; giải ma trận H (3×3) bằng DLT; đếm inliers theo ngưỡng sai số reprojection; chọn H có số inliers lớn nhất và giải lại bình phương tối thiểu trên toàn bộ tập inliers</td>
      <td>N_iter = 1000–2000 (số vòng lặp), τ = 3.0–5.0 pixel (ngưỡng inlier)</td>
      <td>Loại bỏ triệt để các cặp ghép sai (outliers); bước tinh chỉnh (refinement) giúp ma trận H chuẩn xác về mặt hình học, triệt tiêu hiện tượng lệch mép ghép</td>
      <td>Các giả thuyết RANSAC độc lập hoàn toàn → mỗi block/warp thử 1 giả thuyết song song; bước đếm inlier ánh xạ thành bài toán Parallel Reduction; không bị phân kỳ warp như PROSAC</td>
    </tr>
    <tr>
      <td>6. Biến dạng và hòa trộn</td>
      <td>Inverse Warping + Bilinear Interpolation + Distance-weighted Feathering</td>
      <td>Chiếu 4 góc để xác định canvas và ma trận tịnh tiến T; với mỗi pixel đích, chiếu ngược tọa độ về ảnh nguồn qua H⁻¹ (Inverse Warping); nội suy màu từ 4 pixel lân cận (Bilinear); hòa trộn vùng chồng lấp theo tỷ lệ khoảng cách đến viền (Feathering)</td>
      <td>Không có (Phương pháp phi tham số – Parameter-free, tự động thích ứng hoàn toàn theo ma trận H và hình học vùng chồng lấp)</td>
      <td>Inverse Warping loại bỏ hoàn toàn hiện tượng xuất hiện lỗ trống pixel; Bilinear triệt tiêu răng cưa; Feathering khử đường viền nối và làm mượt độ chênh lệch phơi sáng tự nhiên mà không tốn nhiều bộ nhớ như Multi-band</td>
      <td>Mô hình tính toán dạng Gather (1 thread = 1 pixel đầu ra) loại trừ hoàn toàn race condition; Bilinear được hỗ trợ trực tiếp bằng phần cứng Texture Memory của GPU; dễ gộp nhân (Kernel Fusion) để giảm đọc/ghi Global Memory</td>
    </tr>


  </tbody>
</table>


### 3.1 Giai đoạn 1 – Tiền xử lý: Grayscale cố định + Separable Gaussian Blur

#### Cơ chế hoạt động

**Bước 1 – RGB → Grayscale (công thức cố định BT.601):**

```
Gray = 0.299·R + 0.587·G + 0.114·B
```

Trọng số cố định → mỗi pixel RGB được xử lý **độc lập hoàn toàn**, không có sự phụ thuộc dữ liệu giữa các pixel.

**Bước 2 – Separable Gaussian Blur (tách kernel 2D thành 2 lần nhân chập 1D):**

Tận dụng tính chất phân ly (separability) của kernel Gaussian:

```
G(x,y) = g(x) · g(y)  →  Pass 1: nhân chập 1D theo hàng
                         Pass 2: nhân chập 1D theo cột
```

Độ phức tạp giảm từ **O(k²·N)** xuống **O(2k·N)** với N là tổng số pixel và k là kích thước kernel.

#### Tham số chính

| Tham số | Ý nghĩa | Giá trị đề xuất |
|---------|---------|------------------|
| σ | Độ lệch chuẩn Gaussian, kiểm soát mức độ làm mịn | 1.0 – 1.6 |
| k | Kích thước kernel, k = 2⌈3σ⌉+1 | 7 (với σ = 1.0) |

#### Lý do phù hợp với song song hóa

- **Grayscale:** Mỗi pixel độc lập → map trực tiếp `1 thread = 1 pixel`, không cần đồng bộ.
- **Pass 1D ngang:** Mỗi hàng (row) độc lập → song song hóa theo hàng.
- **Pass 1D dọc:** Mỗi cột (column) độc lập → tương tự.
- **Cache-friendly:** Đọc bộ nhớ tuần tự 1D thay vì nhảy cách 2D → giảm cache miss đáng kể.


### 3.2 Giai đoạn 2 – Harris Corner + NMS + Grid-based Top-K

Nhóm chốt **Harris Corner** làm detector duy nhất cho V0–V3 để bảo đảm phép đo speedup so sánh cùng thuật toán và tham số. 

#### Cơ chế hoạt động

**Bước 1 – Tính gradient:** Dùng Sobel 3×3 để tính mức biến thiên cường độ theo hai hướng `Ix` và `Iy` tại mỗi pixel.

**Bước 2 – Xây dựng structure tensor:** Tính ba tích `Ix²`, `Iy²`, `IxIy`, sau đó làm mượt bằng Gaussian để thu `Sxx`, `Syy`, `Sxy`. Các giá trị này tạo ma trận mô tả mức biến thiên cường độ theo hai hướng:

```
M = [[Sxx, Sxy], [Sxy, Syy]]
```

**Bước 3 – Tính Harris response:**

```
R = det(M) - k·trace(M)²
```

`R` lớn và dương biểu thị ứng viên góc; `R` âm thường là cạnh; `R` gần 0 là vùng phẳng.

**Bước 4 – Lọc và chọn keypoint:** Giữ các điểm thỏa `R > α·max(R)`, áp dụng NMS để chỉ giữ cực đại cục bộ, rồi dùng Grid-based Top-K để giới hạn số lượng và phân bố keypoint đều trên ảnh.

#### Tham số chính

| Tham số | Ý nghĩa | Giá trị đề xuất |
|---------|---------|------------------|
| `k` | Độ nhạy của Harris response với cạnh | 0.04 |
| `σ_w` | Độ lệch chuẩn Gaussian khi làm mượt structure tensor | 1.0 – 1.5 |
| Tensor window | Kích thước cửa sổ làm mượt tensor | 3×3 hoặc 5×5 |
| `α` | Hệ số threshold tương đối trong `R > α·max(R)` | 0.01 |
| NMS window | Cửa sổ giữ cực đại cục bộ | 3×3 |
| Grid cell | Kích thước ô dùng cho Spatial Top-K | 32×32 pixel |
| K/cell | Số keypoint tối đa giữ lại trong mỗi ô | Chốt bằng thực nghiệm V0 |

#### Lý do phù hợp với song song hóa

- **Sobel, tensor và Harris response:** Mỗi pixel được ánh xạ trực tiếp thành `1 thread = 1 pixel`; luồng điều khiển đều, ít phân kỳ warp.
- **Stencil cục bộ:** Các pixel lân cận được tái sử dụng bằng shared-memory tiling có halo, giúp giảm số lần đọc global memory.
- **Threshold và NMS:** Mỗi thread kiểm tra response của pixel mình trên response map đã hoàn tất, không có phụ thuộc ghi giữa các pixel.
- **Tìm `max(R)` và thu thập keypoint:** Có thể dùng các mẫu song song chuẩn như reduction, prefix-sum compaction hoặc warp-level aggregation.
- **Grid-based Top-K:** Mỗi ô ảnh có thể được một CUDA block xử lý độc lập, vừa song song hóa chọn điểm vừa bảo đảm phân bố không gian.
- **Khả năng tối ưu theo phiên bản:** V1 dùng global memory; V2 dùng shared-memory tiling; V3 tối ưu compaction và Spatial Top-K dựa trên bottleneck đo được.

Các tham số được chốt sau thí nghiệm V0 và giữ nguyên trong V1–V3.


### 3.3 Giai đoạn 3 – Mô tả điểm đặc trưng: ORB descriptor (BRIEF có xoay, 256 bit)

Nhóm chọn **ORB descriptor** (BRIEF có xoay theo hướng keypoint) để mô tả vùng lân cận của mỗi keypoint bằng một chuỗi nhị phân 256 bit.

#### Cơ chế hoạt động

- **Bước 1 – Lấy vùng lân cận (patch):** Với mỗi keypoint từ Giai đoạn 2, xét patch kích thước $S \times S$ quanh keypoint trên ảnh xám đã làm mịn ở Giai đoạn 1. Làm mịn trước là cần thiết vì mỗi bit chỉ dựa trên so sánh hai pixel, rất dễ bị nhiễu làm sai.
- **Bước 2 – Tính hướng keypoint (Intensity Centroid):** Tính các moment của patch trong vùng tròn bán kính $r$:
  $$m_{pq} = \sum_{x,y} x^p y^q \, I(x, y)$$
  Hướng của keypoint là hướng từ tâm patch tới trọng tâm cường độ:
  $$\theta = \operatorname{atan2}(m_{01}, m_{10})$$
  Khi ảnh bị xoay, trọng tâm cường độ xoay theo, nên $\theta$ cho biết patch đang lệch bao nhiêu so với hướng chuẩn.
- **Bước 3 – Xoay mẫu so sánh (Steering):** Tập $n$ cặp điểm cố định $\{(\mathbf{x}_i, \mathbf{y}_i)\}$ quanh tâm patch được xoay theo $\theta$ bằng ma trận quay $R_\theta$:
  $$(\mathbf{x}_i^\theta, \mathbf{y}_i^\theta) = (R_\theta \mathbf{x}_i,\; R_\theta \mathbf{y}_i)$$
  Góc $\theta$ được rời rạc hóa theo bước 12°, nên có thể tính sẵn các mẫu đã xoay và chỉ cần tra bảng.
- **Bước 4 – Phép thử nhị phân (Binary Test):** Mỗi cặp điểm cho ra 1 bit:
  $$\tau(\mathbf{x}_i, \mathbf{y}_i) = \begin{cases} 1 & \text{nếu } I(\mathbf{x}_i^\theta) < I(\mathbf{y}_i^\theta) \\ 0 & \text{ngược lại} \end{cases}$$
  Ghép $n = 256$ bit lại thành descriptor 32 byte của keypoint.

#### Tham số chính

| Tham số | Ý nghĩa | Giá trị đề xuất |
|---------|---------|------------------|
| `S` (patch size) | Kích thước vùng lân cận dùng để tính hướng và so sánh | 31×31 |
| `r` | Bán kính vùng tròn khi tính intensity centroid | 15 pixel |
| `n` | Số cặp điểm so sánh, tức độ dài descriptor | 256 bit |
| Bước góc | Độ rời rạc hóa hướng $\theta$ khi xoay mẫu | 12° (30 mức) |

#### Lý do phù hợp với song song hóa (GPU)

- **Mỗi keypoint độc lập (Embarrassingly Parallel):** Việc tính descriptor của một keypoint không phụ thuộc keypoint khác, mỗi thread có thể xử lý một keypoint.
- **Khối lượng việc đồng đều:** Mọi keypoint đều tính moment trên cùng một vùng tròn và thực hiện đúng 256 phép so sánh, nên các thread trong warp đi cùng một luồng điều khiển, ít phân kỳ.
- **Mẫu cố định trong Constant Memory:** Tập cặp điểm (kể cả các bản đã xoay) là hằng số, dùng chung cho mọi keypoint. Khi các thread trong warp cùng đọc một cặp điểm, Constant Memory phát (broadcast) giá trị cho cả warp trong một lần truy cập.
- **Đầu ra nhị phân gọn:** Mỗi descriptor chỉ 32 byte (so với 512 byte của SIFT), giảm lượng dữ liệu phải đọc ở Giai đoạn 4 và cho phép so khớp chỉ bằng phép XOR và đếm bit.

### 3.4 Giai đoạn 4 – Ghép cặp điểm: Brute-force Hamming + Lowe Ratio Test + Cross-check

Nhóm chọn **so khớp vét cạn (Brute-force) theo khoảng cách Hamming**, kết hợp hai bộ lọc **Lowe ratio test** và **cross-check** để loại các cặp ghép sai trước khi đưa vào RANSAC.

#### Cơ chế hoạt động

- **Bước 1 – Khoảng cách Hamming:** Với hai descriptor nhị phân $a, b$, khoảng cách là số bit khác nhau:
  $$d_H(a, b) = \operatorname{popcount}(a \oplus b), \quad 0 \le d_H \le 256$$
- **Bước 2 – Tìm láng giềng gần nhất (Brute-force):** Với mỗi descriptor $a_i$ của ảnh A, tính khoảng cách tới **mọi** descriptor $b_j$ của ảnh B, giữ lại láng giềng gần nhất $j^*$ cùng hai khoảng cách nhỏ nhất:
  $$d_1 = d_H(a_i, b_{j^*}), \qquad d_2 = \min_{j \ne j^*} d_H(a_i, b_j)$$
- **Bước 3 – Lowe ratio test:** Chỉ chấp nhận cặp $(i, j^*)$ nếu láng giềng gần nhất tốt hơn hẳn láng giềng thứ hai:
  $$\frac{d_1}{d_2} < \rho$$
  Ở vùng có hoa văn lặp lại (cửa sổ, gạch lát), $d_1 \approx d_2$ nên cặp bị loại vì không đủ phân biệt.
- **Bước 4 – Cross-check (Mutual Nearest Neighbor):** Cặp $(i, j^*)$ chỉ được giữ nếu $a_i$ cũng là láng giềng gần nhất của $b_{j^*}$ khi tìm theo chiều ngược lại B → A. Bước này loại các trường hợp nhiều điểm của A cùng trỏ vào một điểm của B.

Kết quả là tập cặp điểm tương ứng $\{(\mathbf{p}_i, \mathbf{p}'_{j^*})\}$ làm đầu vào cho Giai đoạn 5.

#### Tham số chính

| Tham số | Ý nghĩa | Giá trị đề xuất |
|---------|---------|------------------|
| `ρ` (ratio) | Ngưỡng của Lowe ratio test; càng nhỏ càng chặt | 0.7 – 0.8 |
| Cross-check | Bật/tắt kiểm tra láng giềng gần nhất hai chiều | Bật |

#### Lý do phù hợp với song song hóa (GPU)

- **Bài toán all-pairs giống nhân ma trận:** Cần tính $N_A \times N_B$ khoảng cách độc lập nhau; mỗi thread có thể xử lý một descriptor của A và duyệt toàn bộ B.
- **Phép tính khoảng cách rất rẻ:** Mỗi khoảng cách chỉ gồm XOR và đếm bit 1 trên 256 bit; GPU hỗ trợ đếm bit bằng phần cứng nên phép tính này rất nhanh.
- **Tái sử dụng dữ liệu cao, hợp với Shared-memory Tiling:** Mỗi descriptor của B được mọi thread đọc lại. Nạp từng khối (tile) descriptor B vào Shared Memory để cả block dùng chung sẽ giảm mạnh số lần đọc Global Memory, tương tự tiled matrix multiplication.
- **Tìm $d_1$, $d_2$ là bài toán Reduction:** Khi chia việc duyệt B cho nhiều thread, kết quả cục bộ được gộp bằng warp shuffle; cross-check có cùng cấu trúc, chỉ đổi vai trò A và B.
- **Không dùng KD-tree / FLANN:** Các cấu trúc cây khiến mỗi thread duyệt một nhánh khác nhau (phân kỳ warp, truy cập bộ nhớ không đều) và chỉ cho kết quả xấp xỉ; brute-force cho kết quả chính xác với luồng thực thi đồng nhất.

### 3.5 Giai đoạn 5 – Ước lượng Homography: RANSAC + 4-point DLT + Inlier Refinement

Nhóm chọn **RANSAC chuẩn kết hợp DLT 4 điểm và bước tinh chỉnh (Refinement)** làm giải thuật ước lượng ma trận biến đổi xạ ảnh $H_{3 \times 3}$.

#### Cơ chế hoạt động

- **Bước 1 – Lấy mẫu ngẫu nhiên tối thiểu:** Chọn ngẫu nhiên 4 cặp điểm tương ứng không đồng tuyến từ tập kết quả so khớp ở Giai đoạn 4.
- **Bước 2 – Giải ma trận Homography giả thuyết ($H$):** Thiết lập hệ phương trình tuyến tính $Ah = 0$ ($A$ kích thước $8 \times 9$, mỗi cặp điểm cho 2 phương trình) và giải qua phép phân tích kỳ dị (SVD) hoặc khử Gauss để tìm ma trận $H$ kích thước $3 \times 3$.
- **Bước 3 – Đánh giá giả thuyết và đếm Inliers:** Chiếu toàn bộ các điểm đặc trưng của ảnh nguồn sang ảnh đích qua $H$, tính sai số hình học (reprojection error):
  $$d(p_i', H p_i) = \sqrt{(x_i' - \hat{x}_i)^2 + (y_i' - \hat{y}_i)^2}$$
  Điểm được tính là inlier nếu sai số $d < \tau$ (ngưỡng khoảng cách).
- **Bước 4 – Cập nhật ma trận tối ưu:** Lặp lại quá trình trên $N$ lần; lưu lại ma trận $H$ sở hữu số lượng inliers lớn nhất.
- **Bước 5 – Tinh chỉnh (Refinement):** Giải lại DLT tổng quát bằng phương pháp bình phương tối thiểu (least squares) trên toàn bộ tập inliers tối ưu để triệt tiêu sai số cục bộ.

#### Tham số chính

| Tham số | Ý nghĩa | Giá trị đề xuất |
|---------|---------|------------------|
| `N_iter` | Số vòng lặp lấy mẫu giả thuyết RANSAC | 1000 – 2000 |
| `τ` (threshold) | Ngưỡng khoảng cách sai số reprojection để chấp nhận là Inlier | 3.0 – 5.0 pixel |
| `inlier_ratio_target` | Tỉ lệ inliers kỳ vọng để dừng sớm (early stopping) | 0.7 – 0.8 |

#### Lý do phù hợp với song song hóa (GPU)

- **Các giả thuyết độc lập (Embarrassingly Parallel):** Mỗi vòng lặp sinh và kiểm tra một giả thuyết $H$ hoàn toàn độc lập với nhau, cho phép phân bổ trực tiếp cho các CUDA Block hoặc Warp chạy song song đồng thời.
- **Không gây phân kỳ luồng (No Warp Divergence):** Khác với PROSAC (không gian mẫu phụ thuộc trạng thái động tăng dần), RANSAC chuẩn có luồng thực thi đồng nhất cho mọi giả thuyết, tận dụng tối đa năng lực xử lý của warp.
- **Đếm Inlier bằng Parallel Reduction:** Bước kiểm tra sai số và đếm số điểm inlier của $M$ cặp điểm được ánh xạ thành bài toán song song hóa reduction trên Shared Memory cực kỳ hiệu quả.


### 3.6 Giai đoạn 6 – Biến dạng và hòa trộn: Inverse Warping + Bilinear Interpolation + Distance-weighted Feathering

Nhóm chọn giải pháp **Inverse Warping kết hợp lấy mẫu song tuyến tính (Bilinear Interpolation)** và kỹ thuật hòa trộn **Distance-weighted Feathering** để tạo ảnh toàn cảnh cuối cùng.

#### Cơ chế hoạt động

- **Bước 1 – Xác định kích thước Canvas và Ma trận tịnh tiến ($T$):**
  - Chiếu 4 góc của ảnh nguồn (ảnh B) sang hệ tọa độ của ảnh tham chiếu (ảnh A) qua ma trận $H_{2 \to 1}$.
  - Xác định bounding box bao phủ toàn bộ 8 góc của hai ảnh: $[x_{min}, x_{max}] \times [y_{min}, y_{max}]$.
  - Kích thước canvas toàn cảnh được tính: $W_{pano} = x_{max} - x_{min}$, $H_{pano} = y_{max} - y_{min}$.
  - Thiết lập ma trận tịnh tiến $T$ với độ dịch chuyển $(t_x, t_y) = (-x_{min}, -y_{min})$ để đưa toàn bộ tọa độ về phần dương của canvas.

- **Bước 2 – Chiếu ngược tọa độ (Inverse Warping):**
  - Với mỗi vị trí nguyên $(x, y)$ trên canvas đích, tính tọa độ thực $(u, v)$ tương ứng trên ảnh gốc thông qua ma trận nghịch đảo:
    $$\begin{bmatrix} u' \\ v' \\ w' \end{bmatrix} = (T \cdot H)^{-1} \begin{bmatrix} x \\ y \\ 1 \end{bmatrix} \implies u = \frac{u'}{w'}, \quad v = \frac{v'}{w'}$$
  - Cơ chế *Inverse Warping* đảm bảo mỗi pixel trên canvas đích luôn tìm được giá trị màu tương ứng, khắc phục triệt để hiện tượng xuất hiện các lỗ trống điểm ảnh (holes) của phương pháp chiếu xuôi (*Forward Warping*).

- **Bước 3 – Lấy mẫu nội suy song tuyến tính (Bilinear Interpolation):**
  - Do tọa độ $(u, v)$ là số thực, giá trị màu được nội suy từ 4 pixel nguyên lân cận bao quanh:
    $$I(u, v) = (1 - \Delta u)(1 - \Delta v)I(u_0, v_0) + \Delta u(1 - \Delta v)I(u_1, v_0) + (1 - \Delta u)\Delta v I(u_0, v_1) + \Delta u \Delta v I(u_1, v_1)$$
    trong đó $\Delta u = u - u_0, \Delta v = v - v_0$. Phép nội suy này giúp ảnh uốn mịn màng và loại bỏ hoàn toàn răng cưa do phép biến đổi phối cảnh gây ra.

- **Bước 4 – Hòa trộn trọng số khoảng cách (Distance-weighted Feathering):**
  - Trong vùng chồng lấn (overlap), tính khoảng cách Euclidean $d_1(x, y)$ và $d_2(x, y)$ từ pixel hiện tại tới đường biên gần nhất của từng ảnh thành phần.
  - Trộn màu tuyến tính theo trọng số khoảng cách:
    $$I_{blend}(x, y) = \frac{d_1(x, y) \cdot I_1(x, y) + d_2(x, y) \cdot I_2(x, y)}{d_1(x, y) + d_2(x, y)}$$
  - Khi tiến gần viền của ảnh 1 thì $d_1 \to 0$, trọng số của ảnh 1 giảm dần về 0 và chuyển tiếp mượt mà sang ảnh 2, giúp xóa nhòa đường nối cắt và cân bằng độ chênh lệch ánh sáng giữa hai góc chụp.

#### Đặc tính tham số: Phương pháp phi tham số (Parameter-free)

Một ưu điểm lớn của sự kết hợp **Inverse Warping + Distance-weighted Feathering** là phương pháp này **hoàn toàn không có tham số cần tinh chỉnh (Parameter-free)**:
- **Tính tất định trong uốn ảnh:** Tọa độ lấy mẫu của từng pixel đầu ra được xác định chính xác và duy nhất thông qua ma trận biến đổi xạ ảnh $H_{2 \to 1}$ đã tìm được ở Giai đoạn 5.
- **Tự động thích ứng trong hòa trộn:** Trọng số trộn màu $\alpha_1, \alpha_2$ tại mỗi điểm được tính toán tự động dựa trên khoảng cách hình học thực tế đến viền ảnh ($d_1, d_2$). Không cần đặt ngưỡng cố định hay bề rộng dải cắt cứng nhắc, giúp pipeline hoạt động ổn định trên mọi cặp ảnh có góc xoay và độ chồng lấp khác nhau.

#### Lý do phù hợp với song song hóa (GPU)

- **Mô hình tính toán dạng Gather (1 thread = 1 pixel output):** Toàn bộ không gian canvas đích được phân rã thành lưới luồng 2D (`1 thread = 1 pixel`). Mỗi luồng chỉ đọc từ ảnh nguồn và ghi đúng vào một vị trí pixel đích, loại bỏ hoàn toàn xung đột ghi bộ nhớ (*race condition*) mà không cần bất kỳ lệnh khóa hay đồng bộ nào.
- **Tận dụng phần cứng Texture Memory:** GPU trang bị bộ điều khiển Texture chuyên dụng có khả năng nội suy Bilinear trực tiếp ở cấp độ phần cứng chỉ trong 1 chu kỳ truy cập, đồng thời sở hữu bộ đệm không gian 2D (Spatial Cache) giúp tăng tốc độ lấy mẫu khi ảnh bị biến dạng hình học.
- **Khả năng gộp nhân (Kernel Fusion):** Toàn bộ quy trình từ tính tọa độ nghịch đảo, lấy mẫu Bilinear cho đến hòa trộn Feathering có thể gộp chung vào duy nhất một CUDA kernel. Nhờ đó, chương trình không phải lưu các bức ảnh uốn trung gian ra bộ nhớ toàn cục (Global Memory), tiết kiệm đáng kể băng thông VRAM.
- **Tiết kiệm tài nguyên so với Multi-band Blending:** Kỹ thuật Feathering có chi phí tính toán nhẹ và độ phức tạp thấp hơn nhiều so với việc phân rã tháp Laplacian nhiều tầng của Multi-band, đảm bảo khả năng mở rộng tốc độ (speedup) tốt trên các tập ảnh phân giải lớn (Full HD, 4K).


---

## 4. Phiên bản tuần tự (V0)

Phiên bản V0 là cài đặt C/C++ **đơn luồng** (không dùng GPU) của toàn bộ pipeline ghép ảnh.

---

## 5. Kết quả và phân tích hiệu năng V0

---

### 5.1 Độ chính xác

### 5.2 Ảnh toàn cảnh đầu ra

### 5.3 Bảng thời gian thực thi theo từng giai đoạn

### 5.4 Phân tích giai đoạn chiếm nhiều thời gian nhất

### 5.5 Ước lượng theo Định luật Amdahl

---

## 6. Bộ dữ liệu

### 6.1 Quy chuẩn ảnh và cách chọn cặp ảnh

Mỗi cặp ảnh được **cắt tỉ lệ 16:9** và resize về 3 độ phân giải:

| Tên | Độ phân giải | Mục đích                          |
|-----|--------------|-----------------------------------|
| Nhỏ | 1280 × 720   | Phát triển nhanh và debug         |
| Vừa | 1920 × 1080  | Benchmark chính                   |
| Lớn | 3840 × 2160  | Kiểm tra khả năng mở rộng của GPU |

### 6.2 Danh sách địa điểm chụp

| STT | Địa điểm    | Bối cảnh      | Phân loại   | Mô tả cặp ảnh   | Đường dẫn |
|-----|-------------|---------------|-------------|-----------------|-----------|
| 1   |             | Indoor        | Standard    |                 |           |
| 2   |             | Outdoor       | Standard    |                 |           |
| 3   |             | Team's choice | Standard    |                 |           |
| 4   |             | Team's choice | Challenging |                 |           |

**Link bộ dữ liệu (Google Drive):** ...

---

## 7. Kế hoạch GPU

### 7.1 Các giai đoạn sẽ chuyển lên GPU

### 7.2 Chi tiết ánh xạ luồng V1

### 7.3 Ý tưởng tối ưu hóa cho V2, V3, ...

1. **[Ý tưởng 1]**

2. **[Ý tưởng 2]**

3. **[Ý tưởng 3]**

---

## Tài liệu tham khảo